# 01: Exploring raw OPM-MEG files

**Who this is for:** new users of the Scully Center MEG lab.

**What you will do:**

1. Copy the sample dataset on the file server to your own partition.
2. Convert the raw `.cMEG` recordings to FIF with **cMEG2FIF**, in your Terminal.
3. Explore the files, compare what Cerca writes with what the converter produces.

## 0. Setup

Choose **one** of the two setups, then set your paths in 0.3:

- **0.1 Option A: Cluster (preferred).** Jupyter runs on scotty and you use it from your laptop's browser. The data never leave the server.
- **0.2 Option B: Local laptop.** Jupyter runs on your laptop, reading the data from your mounted lab partition.

### 0.1 Option A: Cluster setup (preferred)

#### 0.1.1 Pick a port and SSH in with the tunnel:

Choose something in the 4000-9999 range that avoids 8800-8899. On a Mac, also avoid 5000 and 7000.
In your Terminal ssh to scotty:

```ssh -L 4817:localhost:4817 NETID@scotty.pni.princeton.edu```

Check that nobody else is using the port on scotty:
```ss -ltn | grep 4817```


#### 0.1.2 Create the conda environment

Create an environment from scratch:

```bash
conda create -n pygers-meg -c conda-forge python=3.11 mne jupyterlab pandas ipykernel ipympl mne-bids
conda activate pygers-meg
pip install "cmeg2fif[plot] @ git+https://github.com/mrribbits/cMEG2FIF-Scully.git"
pip install bids-validator-deno
```

Or create it using the supplied scotty-environment.yml:
```bash
conda env create -f scotty-environment.yml -n pygers-meg
conda activate pygers-meg
```

Check the converter installed:

```bash
cmeg2fif --version
```

#### 0.1.3 Launch Jupyter on scotty
```
jupyter lab --no-browser --ip=127.0.0.1 --port=4817 --ServerApp.port_retries=0
```

#### 0.1.4 Open on your laptop's browser
Copy the URL that Jupyter prints, which looks like `http://127.0.0.1:4817/lab?token=...`, and paste it into your local browser.

#### 0.1.5 To shut down
Press Ctrl-C twice in the scotty session. If you ever leave one running by accident, `jupyter server list` shows it and `jupyter server stop 4817` stops it.


### 0.2 Option B: Local laptop setup

#### 0.2.1 Python environment (do this once, in your Terminal)

Create an environment:

```bash
conda create -n pygers-meg -c conda-forge python=3.11 mne jupyterlab pandas ipympl mne-bids
conda activate pygers-meg
pip install "cmeg2fif[plot] @ git+https://github.com/mrribbits/cMEG2FIF-Scully.git"
```

Check the converter installed:

```bash
cmeg2fif --version
```

#### 0.2.2 Mount your 'cup' lab partition

All data stay on the server: **do not copy them to your laptop.** Mount the server in Finder (macOS: *Go → Connect to Server…*) or File Explorer (Windows: *Map network drive*).


### 0.3 Copy sample data and set paths

#### 0.3.1 Copy the sample dataset to a folder in your lab's partition

Everyone works on their **own copy**, so the shared folder is never modified. In the cell below:

1. Set `RAW_DIR` to where your copy of the `raw` folder should go, e.g. `/jukebox/<lab>/<personal folder>/auditory-oddball/sub-001/raw`.
2. Run the cell. It creates `RAW_DIR` and copies the whole shared `raw` folder into it.

Re-running is safe: `rsync` only copies files that are new or changed, so a second run finishes almost immediately.

This works as written when Jupyter runs on scotty (Option A). On a laptop (Option B), both paths must be as your laptop sees them through the mounted share (`/Volumes/...` on a Mac), and the `!` commands need macOS or Linux.


In [ ]:
from pathlib import Path

SHARED_RAW = "/jukebox/scratch/mpinsk/MEG/pygers-meg/auditory-oddball/sub-001/raw"  # shared, read-only

# ---- EDIT THIS LINE ------------------------------------------------------
RAW_DIR = Path("/<path>/auditory-oddball/sub-001/raw")  # your copy
# ---------------------------------------------------------------------------

!mkdir -p "{RAW_DIR}"
!rsync -av "{SHARED_RAW}/" "{RAW_DIR}/"


#### 0.3.2 Set the output folder

Edit the `OUT_DIR` line below (use `/jukebox/...` on scotty, `/Volumes/...` on a laptop), then run the cell. `RAW_DIR` comes from 0.3.1.


In [ ]:
from pathlib import Path

# ---- EDIT THIS LINE ------------------------------------------------------
OUT_DIR = Path("/<path>/auditory-oddball/sub-001/fif")  # converter output
# ---------------------------------------------------------------------------
# RAW_DIR was set in 0.3.1

# Recording prefixes = the acquisition start time Cerca put in the file names
ER_PREFIX   = "20260929_125813"   # empty-room (noise) recording
DATA_PREFIX = "20260929_134022"   # task recording (auditory oddball)

ER_DIR    = RAW_DIR / "cMEG" / f"{ER_PREFIX}_cMEG_Data"
DATA_DIR  = RAW_DIR / "cMEG" / f"{DATA_PREFIX}_cMEG_Data"
EINSCAN   = RAW_DIR / "einscan"

XFM_FILE = EINSCAN / "sub-001-head+helmet_dev2head_xfm.tsv"
XYZ_FILE = EINSCAN / "sub-001-head+helmet_digitisation_from_mesh_3_xfmd.xyz"

# Converter outputs (BIDS-style names, so they can drop straight into a BIDS project later)
ER_FIF   = OUT_DIR / "sub-001_ses-001_task-noise_meg.fif"
DATA_FIF = OUT_DIR / "sub-001_ses-001_task-oddballftcued_meg.fif"

LINE_FREQ = 60                    # North American mains

## 1. What's in the raw session? (file inventory)

Cerca's acquisition software writes these files next to each other, sharing the start-time prefix:

| File | Contents |
|---|---|
| `<prefix>_meg_NNN.cMEG` | Raw data, split into numbered parts |
| `<prefix>_meg.json` | Cerca's sidecar: sampling rate, duration, a comment (Section 2) |
| `<prefix>_channels.tsv` | Channel names, types, gains (V/nT), good/bad status |
| `<prefix>_HelmConfig.tsv` | Sensor positions and orientations in the helmet |
| `<prefix>_SensorTransform.tsv` | Cerca's default sensor transform (we use the cReg `--xfm` instead) |
| `<prefix>_SessionInfo.txt` | Session notes from the acquisition software |

The other folders: `einscan/` (coregistration outputs from cReg), `behav/` (PsychoPy logs), `eye/` (EyeLink), `mri/` (DICOMs).

## 2. The original `_meg.json` sidecar

Cerca writes a `_meg.json` next to each recording. It *looks* like a BIDS sidecar. The next cell lists every field and its value, for the task recording and the empty room side by side.

In [ ]:
import json                        # read the Cerca _meg.json sidecars
import re                          # regular expressions: pick apart file names and log lines
import platform                    # report the Python version
from collections import Counter    # count things, e.g. how many channels of each type

import numpy as np                 # arrays and maths
import pandas as pd                # tables: TSV/CSV files, summaries
import matplotlib.pyplot as plt    # plots
import mne                         # MNE-Python: reads and analyses the FIF files

# MNE prints a lot of progress messages by default. "WARNING" hides those and shows
# only warnings and errors. Use "INFO" to see everything MNE is doing.
mne.set_log_level("WARNING")

# By default pandas shortens long tables (showing "..." in the middle) and long text.
# These show up to 300 rows and up to 120 characters per cell, so whole channel lists fit.
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 120)

print(f"Python {platform.python_version()}  |  MNE {mne.__version__}  |  "
      f"NumPy {np.__version__}  |  pandas {pd.__version__}")

js_data = json.loads((DATA_DIR / f"{DATA_PREFIX}_meg.json").read_text())
js_er   = json.loads((ER_DIR   / f"{ER_PREFIX}_meg.json").read_text())

print(f"{'field':28s} {'task':>14s}   {'empty room':>14s}")
for key in js_data:
    print(f"{key:28s} {js_data[key]!r:>14}   {js_er.get(key)!r:>14}")

**What's wrong with it?** It is not valid BIDS:

- **Descriptive fields** (`InstitutionName`, `Manufacturer`, `ManufacturersModelName`, `SoftwareVersions`, `TaskName`, …) are `"n/a"`, and `TaskDescription` is empty.
- **`PowerLineFrequency`** is `0`. BIDS requires the mains frequency in Hz (60 in the US, 50 in Europe), or `"n/a"`.
- **`RecordingType`** is `"Measurement"` (task) or `"Noise"` (empty room). BIDS allows only `"continuous"`, `"epoched"` or `"discontinuous"`.
- **Channel counts** don't match the real channel layout: the reference sensors are counted in `MEGChannelCount`, and `MEGREFChannelCount` is `0` (Section 2.1).
- **`DigitizedLandmarks`** and **`DigitizedHeadPoints`** are `false`, even for the task recording, which has a head scan.

### 2.1 Where are the reference sensors?

Cerca's `channels.tsv` uses only **three** channel types: `MEGMAG`, `TRIG` and `MISC`. But the system really records **four** kinds of channels. The two **reference sensors**, 3N and 3O, are incorrectly labelled `MEGMAG` just like the on-head sensors.

Cerca's _meg.json follows the same view. It does have a `MEGREFChannelCount` field, but sets it to `0` and counts the reference channels inside `MEGChannelCount` (198 instead of 192).

Nothing in `channels.tsv` marks a sensor as a reference. The clue is in `HelmConfig.tsv`, which lists only the sensors that sit in a **helmet slot**. A sensor that is recorded but has no slot must be a reference. The next cell makes that comparison, which is exactly what cMEG2FIF does during conversion when it labels these channels `ref_meg` in the FIF.

In [ ]:
def read_tsv(path):
    # Tab-separated file -> table. Everything kept as text; "n/a" stays "n/a"; "#" lines are comments.
    return pd.read_csv(path, sep="\t", comment="#", dtype=str, keep_default_na=False)

ch_tsv = read_tsv(DATA_DIR / f"{DATA_PREFIX}_channels.tsv")
helm   = read_tsv(DATA_DIR / f"{DATA_PREFIX}_HelmConfig.tsv")

print("channels.tsv channel types/counts:")
for ch_type, n in ch_tsv["type"].value_counts().items():
    print(f"  {ch_type:22s} {n}")

print("\n_meg.json channel types/counts:")
for key, val in js_data.items():
    if key.endswith("ChannelCount"):
        print(f"  {key:22s} {val}")

print(f"\nHelmConfig.tsv: {len(helm)} sensors in helmet slots")

## 3. Convert the recordings with cMEG2FIF

The converter reads every numbered part (`_meg_001.cMEG`, `_meg_002.cMEG`, …), joins them, attaches sensor geometry, decodes triggers and buttons, and writes a FIF plus a conversion log. See the [cMEG2FIF README](https://github.com/mrribbits/cMEG2FIF-Scully) for the full option list.

Two conversions:

- **Empty room** (`task-noise`): no head was in the helmet, so **no** `--xfm` / `--dig`.
- **Task recording** (`task-oddballftcued`): with `--xfm` (device→head transform) and `--dig` (head-shape points and fiducials) from the Einscan / cReg coregistration.

The next cell runs both conversions right here in the notebook. Before each one it prints the exact command being run, then shows the converter's output as it goes. The task recording takes several minutes.

- `--no-plot` is always included: the converter's pop-up 3D window can't open from inside a notebook.
- The converter never overwrites existing output. If a FIF is already there, the cell skips that recording. To convert again, set `FORCE = True` (this adds `--force`). If you reconvert after Section 4 has opened the FIF files, run Section 4's loading cell again so MNE reads the new files.

In [ ]:
import shlex, shutil, subprocess, sys

FORCE = False   # True = convert again even if the FIF already exists (adds --force)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# The cmeg2fif command and peripherals file installed in this conda environment
CMEG2FIF = shutil.which("cmeg2fif") or str(Path(sys.executable).parent / "cmeg2fif")
PERIPHERALS = Path(sys.prefix) / "share" / "cmeg2fif" / "cMEG_peripherals.tsv"

conversions = [
    ("Empty-room recording", ER_FIF,
     [CMEG2FIF, ER_DIR / f"{ER_PREFIX}_meg_001.cMEG", "--out", ER_FIF,
      "--peripherals", PERIPHERALS, "--line-freq", LINE_FREQ, "--no-plot"]),
    ("Task recording (with head coregistration)", DATA_FIF,
     [CMEG2FIF, DATA_DIR / f"{DATA_PREFIX}_meg_001.cMEG", "--out", DATA_FIF,
      "--peripherals", PERIPHERALS, "--line-freq", LINE_FREQ,
      "--xfm", XFM_FILE, "--dig", XYZ_FILE, "--no-plot"]),
]

for label, fif, cmd in conversions:
    cmd = [str(x) for x in cmd] + (["--force"] if FORCE else [])
    print(f"===== {label}")
    print("$ " + " ".join(shlex.quote(x) for x in cmd) + "\n")     # the exact command
    if fif.exists() and not FORCE:
        print(f"{fif.name} already exists, skipping (set FORCE = True to convert again)\n")
        continue
    # Run the converter and print its output line by line as it works
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"cmeg2fif failed for {label} (exit code {proc.returncode})")
    print()

## 4. Does the FIF hold the correct values for BIDS?

**Important:** cMEG2FIF does **not** write a `_meg.json`. Its output is a FIF file (plus the log). The sidecar is written later, by the **BIDS conversion step**, which reads its values *from the FIF*.

So the question is really: does the FIF hold the correct values, so that the BIDS step will write a valid sidecar? Load the FIF files (only the headers are read, not the data), look at everything the header holds, then pick out the fields the BIDS step will use.

In [ ]:
raw    = mne.io.read_raw_fif(DATA_FIF, preload=False)
raw_er = mne.io.read_raw_fif(ER_FIF, preload=False)
print(raw)
print(raw_er)

**What's in a FIF header?** `raw.info` holds everything stored about the recording apart from the data itself: sampling rate, mains frequency, recording date, channel list, digitisation points, device-to-head transform, bad channels, and more. Displaying it shows every field that has a value:

In [ ]:
# The FIF header: every non-empty field and its value
raw.info

Some fields are empty for this recording and so don't appear above. To see every field name a FIF header can hold:

In [ ]:
# All field names in the header, including empty ones
print(sorted(raw.info.keys()))

Now pick out the fields that correspond to the problems in Cerca's `_meg.json` (Section 2). 

Just note that one of the values below doesn't come from `raw.info`: the **data object** (what kind of MNE object the file opened as).

In [ ]:
# Count the channels of each type, e.g. {"mag": 192, "ref_meg": 6, "stim": 20, "misc": 6}
types = Counter(raw.get_channel_types())

# Digitisation points stored in the FIF (fiducials + head shape). "or []" gives an
# empty list instead of None when there are none, e.g. in the empty-room file.
dig = raw.info["dig"] or []

# Each line prints a label padded to 26 characters (":26s") so the values line up.

# The kind of MNE object the file opened as. read_raw_fif always gives a continuous
# Raw object; there is no "recording type" field in a FIF.
print(f"{'data object':26s} {type(raw).__name__}   (continuous recording)")

# Mains frequency in Hz, set by cMEG2FIF (--line-freq). Cerca's JSON had 0.
print(f"{'line_freq (Hz)':26s} {raw.info['line_freq']}")

# Number of channels of each type (0 if that type isn't present)
for t in ["mag", "ref_meg", "stim", "misc"]:
    print(f"{'channels: ' + t:26s} {types.get(t, 0)}")

# Free-text recording comment, copied from Cerca's TaskDescription.
# "!r" shows the raw value, so an empty string appears as '' rather than nothing.
print(f"{'description':26s} {raw.info['description']!r}")

# Count digitisation points by kind: 1 = fiducial (nasion, LPA, RPA), 4 = head-shape point
print(f"{'fiducials':26s} {sum(d['kind'] == 1 for d in dig)}")
print(f"{'head-shape points':26s} {sum(d['kind'] == 4 for d in dig)}")


**What's fixed after running cMEG2FIF, compared with Cerca's `_meg.json` (Section 2):**

- **Line frequency** (`line_freq`) is **60 Hz** instead of `0` (the converter's `--line-freq`, default 60).
- **Recording type:** a FIF has no recording-type field. Instead, the file opens as a continuous `Raw` object, and the future BIDS step writes `RecordingType: "continuous"` for it, replacing Cerca's `"Measurement"` / `"Noise"`.
- **Channel counts:** the 6 reference channels are now their own type (`ref_meg`), so `mag` is 192 rather than 198. The stim channels include the decoded `STI101` and `STI_BTN`, and the button inputs moved from misc to stim.
- **Digitisation:** the task FIF holds the fiducials and head-shape points from the head scan, so `DigitizedLandmarks` and `DigitizedHeadPoints` will be `true`.

**Not fixed:**

- The **descriptive fields** (institution, manufacturer, model, …) are not stored in a FIF. They have to be supplied at the future BIDS step.
- **`description`** is empty because Cerca's `TaskDescription` was empty. Typing a comment into the cAcq software *during the session* would fill both.

## 5. Comparing raw `channels.tsv` to FIF channels

Let's count and compare the channels.

In [ ]:
tsv_names, fif_names = set(ch_tsv["name"]), set(raw.ch_names)
print(f"channels.tsv: {len(tsv_names)}   FIF: {len(fif_names)}")
print("\nIn channels.tsv but not (by that name) in the FIF:")
print("   ", sorted(tsv_names - fif_names, key=lambda s: (len(s), s)))
print("\nIn the FIF but not in channels.tsv:")
print("   ", sorted(fif_names - tsv_names))
print("\nReference channels in the FIF:", sorted(n for n, t in zip(raw.ch_names, raw.get_channel_types())
                                              if t == "ref_meg"))
print("\nThe renamed BNC inputs are explained by the peripherals file. "
      "STI101 and STI_BTN are the two channels the converter creates.")

## 6. Bad channels

Bad channels are stored in `raw.info['bads']`. MNE excludes them from most processing. cMEG2FIF copies any channel marked `bad` in Cerca's `channels.tsv` into this list.

In [ ]:
print("Task recording  info['bads']:", raw.info["bads"])
print("Empty room      info['bads']:", raw_er.info["bads"])
print("\nchannels.tsv 'status' values:")
print(ch_tsv["status"].value_counts().to_string())

If every status is `undetermined`, nobody marked channels during acquisition, so `info['bads']` is empty. That does **not** mean every channel is good. It's your job to look.



### 6.1 Find candidate bad channels

A quick screen on one minute of data: flag MEG channels that are **flat** (tiny variance) or **noisy** (variance far above the others). These are only *candidates*. Always confirm by looking at the traces and spectra before marking anything.

In [ ]:
def screen_channels(r, tmin=30.0, dur=60.0, z_thresh=5.0):
    tmax = min(tmin + dur, r.times[-1])
    tmin = max(0.0, tmax - dur)
    seg = r.copy().crop(tmin, tmax).pick(["mag", "ref_meg"]).load_data()
    seg.filter(1.0, 40.0, verbose=False)                   # ignore drift and line noise
    sd = seg.get_data().std(axis=1)
    log_sd = np.log10(sd + 1e-30)
    med = np.median(log_sd)
    mad = np.median(np.abs(log_sd - med)) * 1.4826 + 1e-12
    z = (log_sd - med) / mad
    df = pd.DataFrame({"channel": seg.ch_names, "sd_fT": sd * 1e15, "robust_z": z})
    df["flag"] = np.where(z > z_thresh, "noisy", np.where(z < -z_thresh, "flat", ""))
    return df.sort_values("robust_z", key=np.abs, ascending=False)

scr = screen_channels(raw)
cands = scr[scr["flag"] != ""]
print(f"{len(cands)} candidate bad channel(s) in the task recording (1–40 Hz, robust z > 5):")
display(cands if len(cands) else scr.head(10))

To mark channels as bad for your analysis (in memory; this doesn't change the file on disk):

```python
raw.info["bads"] = ["2A [X]", "2A [Y]"]       # example names
```

On OPMs a problem usually affects a whole sensor, so check all three axes of a suspect sensor. You can also mark bads interactively by clicking traces in `raw.plot()`.

## 7. The conversion log

Every run writes `<output>_conversion_log.txt` next to the FIF. It's the **provenance record**. Keep it with the data.

This cell just prints out the most important lines.

In [ ]:
def parse_log(text):
    info = {}
    for key, pat in [("version", r"cMEG2fif version (\S+)"), ("run", r"Run:\s+(.+)"),
                     ("host", r"Host:\s+(.+)"), ("versions", r"Versions:\s+(.+)"),
                     ("output", r"Output:\s+(.+)"), ("peripherals", r"Peripherals:\s+(.+)"),
                     ("helmet", r"Helmet:\s+(.+)"), ("start", r"Recording start.*?:\s+(.+)")]:
        m = re.search(pat, text)
        info[key] = m.group(1).strip() if m else None
    info["summary"] = next((l.strip() for l in text.splitlines() if "MEG channels," in l), None)
    info["used_lines"] = [l.strip() for l in text.splitlines()
                          if re.match(r"\s*bit\s+\d+", l) and "unused" not in l]
    info["unused_lines"] = [l.strip() for l in text.splitlines()
                            if re.match(r"\s*bit\s+\d+", l) and "unused" in l]
    info["warnings"] = [l.strip() for l in text.splitlines()
                        if re.search(r"warn|error|mismatch|jump|incomplete", l, re.I)]
    info["bads"] = [l.strip() for l in text.splitlines() if re.search(r"\bbad", l, re.I)]
    return info

def log_path_for(fif):
    # sub-001_..._meg.fif -> sub-001_..._meg_conversion_log.txt (written next to the FIF)
    return fif.with_name(fif.stem + "_conversion_log.txt")

log_text = log_path_for(DATA_FIF).read_text()
log = parse_log(log_text)
for k in ["version", "run", "host", "versions", "output", "helmet", "peripherals", "start", "summary"]:
    print(f"{k:12s} {log[k]}")
print("\nLines that carried a signal:")
for l in log["used_lines"]:
    print("   ", l)
print("\nLines that never changed (unused):")
for l in log["unused_lines"]:
    print("   ", l)
print("\nWarnings / problems:", log["warnings"] or "none")
print("Bad-channel lines:  ", log["bads"] or "none")

**Questions to answer from the log:**

1. Which VPixx trigger bits were used? What's the largest code `STI101` could contain?
2. Which response buttons did the participant actually press?

## 8. Peripherals, triggers and buttons

### 8.1 The peripherals file

Cerca labels its analog inputs only as `BNC 1` … `BNC 16`. The **peripherals file** (`cMEG_peripherals.tsv`) says what is plugged into each one, what the converter should name it, and what the converter should **do** with it. Its `type` column is an instruction to cMEG2FIF, not an MNE channel type. At our facility it has two values:

| `type` in the peripherals file | What cMEG2FIF does | MNE channel type in the FIF |
|---|---|---|
| `button` | keeps the line **and** adds it as a bit to `STI_BTN` (first button = 1, then 2, 4, 8, …) | `stim` |
| `misc` | keeps the voltage as recorded (e.g. the eye tracker's analog output) | `misc` |

MNE has no "button" channel type, so button inputs become `stim` channels. The three eye tracking channels and the remaining 3 BNC inputs that are not listed in the file are kept as `misc`.

The cell below loads the peripherals file the converter used (its path is in the conversion log).

In [ ]:
periph_path = Path(log["peripherals"])      # from the conversion log (Section 7)
print("Peripherals file used by the converter:", periph_path)

### 8.2 Buttons → `STI_BTN`

Each button line becomes one bit of `STI_BTN`, in the order listed in the peripherals file: `R_thumb`=1, `R_index`=2, `R_middle`=4, … `L_pinky`=512. The value tells you **which** button was pressed. If two buttons are held at once, their bits add (e.g. `R_index` + `L_index` = 2 + 64 = 66).

In [ ]:
# Load the peripherals file (path from Section 8.1); its "button" rows set the STI_BTN bits
periph = read_tsv(periph_path)

trig_names = [c for c in raw.ch_names if c.startswith("Trigger")]
btn_bits = {}
if periph is not None:
    btn_names = list(periph.loc[periph["type"].str.lower() == "button", "name"])
    btn_bits = {n: 2 ** i for i, n in enumerate(btn_names)}
btn_names = [n for n in btn_bits if n in raw.ch_names]
analog_names = [c for c, t in zip(raw.ch_names, raw.get_channel_types()) if t == "misc"]
print("Trigger lines :", trig_names)
print("Button lines  :", btn_names)
print("Analog (misc) :", analog_names)

events = mne.find_events(raw, stim_channel="STI101", consecutive=True, verbose=False)
buttons = mne.find_events(raw, stim_channel="STI_BTN", verbose=False)
print(f"\nSTI101: {len(events)} events   STI_BTN: {len(buttons)} presses")

def plot_window(r, picks, center_sample, pre=0.5, post=8.5, combined=None, title=""):
    # Plot a few seconds of the `picks` channels around `center_sample`,
    # with the decoded code channel (`combined`) as a step plot underneath.
    sf = r.info["sfreq"]
    start = max(int(center_sample - r.first_samp - pre * sf), 0)
    stop = min(int(center_sample - r.first_samp + post * sf), r.n_times)
    data, times = r.get_data(picks=picks, start=start, stop=stop, return_times=True)
    n = len(picks) + (1 if combined else 0)
    fig, axes = plt.subplots(n, 1, figsize=(10, 0.55 * n + 1.2), sharex=True)
    axes = np.atleast_1d(axes)
    # Same y-range for every line, so a line that never switches stays flat
    # instead of being stretched to fill its panel
    lo, hi = data.min(), data.max()
    pad = 0.05 * (hi - lo)
    for ax, name, y in zip(axes, picks, data):
        ax.plot(times, y, lw=0.8)
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=8)
        ax.set_yticks([])
    if combined:
        c, t = r.get_data(picks=[combined], start=start, stop=stop, return_times=True)
        axes[-1].step(t, c[0], where="post", color="k", lw=1.2)
        axes[-1].set_ylabel(combined, rotation=0, ha="right", va="center", fontweight="bold")
        for val in np.unique(c[0][c[0] > 0])[:10]:
            axes[-1].text(t[np.argmax(c[0] == val)], val, f" {int(val)}", fontsize=8, va="bottom")
    axes[-1].set_xlabel("time (s)")
    fig.suptitle(f"{title}\n(all lines on the same scale: {lo:.1f} to {hi:.1f} V)")
    fig.tight_layout()
    plt.show()

if len(buttons):
    plot_window(raw, btn_names, buttons[0, 0], pre=1.0, post=120.0, combined="STI_BTN",
                title="Button lines (V) and the decoded STI_BTN code")
else:
    print("No button presses in this recording.")

### 8.3 Analog eye channels

`eye_x`, `eye_y` and `eye_pupil` carry the EyeLink's analog output (in volts). BNC 14–16 are free inputs, so they should be flat. (The EyeLink `.EDF` file itself will be covered later)

In [ ]:
start = int(min(3000 * raw.info["sfreq"], raw.n_times // 2))
stop = min(start + int(20 * raw.info["sfreq"]), raw.n_times)
data, times = raw.get_data(picks=analog_names, start=start, stop=stop, return_times=True)
fig, axes = plt.subplots(len(analog_names), 1, figsize=(10, 1.1 * len(analog_names) + 1), sharex=True)
for ax, name, y in zip(np.atleast_1d(axes), analog_names, data):
    ax.plot(times, y, lw=0.6)
    ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=8)
    ax.text(1.0, 0.5, f" range {np.ptp(y):.3f} V", transform=ax.transAxes, fontsize=8, va="center")
np.atleast_1d(axes)[-1].set_xlabel("time (s)")
fig.suptitle("Analog BNC inputs, 20 s of the task recording")
fig.tight_layout()
plt.show()

### 8.4 VPIXX Triggers → `STI101`

The 8 VPixx trigger lines are recorded as voltages. Each line is one **bit** of the stimulus code. The converter thresholds each line and adds them up: `STI101 = 1·T1 + 2·T2 + 4·T3 + … + 128·T8`.

The plot shows a few seconds around the first stimulus code: the raw trigger voltages on top, the combined `STI101` code underneath.

In [ ]:
if len(events):
    plot_window(raw, trig_names, events[0, 0], combined="STI101",
                title="VPixx trigger lines (V) and the decoded STI101 code")

## 9. A first look at signal quality

### 9.1 Spectra: empty room vs. task

The empty room shows the environment's noise floor with nobody present. Compare it with the task recording. Look for:

- **60 Hz and harmonics** (120, 180 Hz): mains interference
- **Low-frequency rise** below a few Hz: drift, movement, residual fields
- **Alpha (~10 Hz)** in the task data but not the empty room: brain signal

Each spectrum uses one minute of data (only that minute is read from disk).

In [ ]:
def mean_psd(r, tmin=30.0, dur=60.0, fmax=200.0):
    tmax = min(tmin + dur, r.times[-1])
    tmin = max(0.0, tmax - dur)
    seg = r.copy().crop(tmin, tmax).pick("mag", exclude="bads").load_data()
    psd = seg.compute_psd(fmin=0.5, fmax=fmax, n_fft=int(4 * seg.info["sfreq"]), verbose=False)
    amp = np.sqrt(psd.get_data()) * 1e15                         # fT / sqrt(Hz)
    return psd.freqs, np.median(amp, axis=0), np.percentile(amp, [10, 90], axis=0)

plt.figure(figsize=(10, 4.5))
for label, r, c in [("empty room", raw_er, "tab:gray"), ("task", raw, "tab:blue")]:
    f, med, (lo, hi) = mean_psd(r)
    plt.semilogy(f, med, color=c, label=f"{label} (median over sensors)")
    plt.fill_between(f, lo, hi, color=c, alpha=0.15)
for h in (60, 120, 180):
    plt.axvline(h, color="r", ls=":", lw=0.8)
plt.xlabel("frequency (Hz)"); plt.ylabel("amplitude (fT/√Hz)")
plt.title("OPM noise spectra (shaded: 10th–90th percentile across sensors)")
plt.legend(); plt.tight_layout(); plt.show()

### 9.2 Browse the raw data

`raw.plot()` is MNE's data browser. `%matplotlib widget` (from `ipympl`, installed in Section 0) makes it interactive inside the notebook, which works both on scotty and on your laptop. Click inside the figure first so it has keyboard focus, then:

- **← / →** scroll in time
- **↑ / ↓** scroll through channels
- **Click a trace** to mark that channel bad (written to `raw.info["bads"]` when you close the figure with the × in its toolbar)

On scotty, every redraw travels over the tunnel, so keep `duration` and `n_channels` small if it feels sluggish.

In [ ]:
%matplotlib widget
import mne

mne.viz.set_browser_backend("matplotlib")
raw.plot(start=60, duration=10, n_channels=30, events=events, # raw = the task recording, opened in Section 4.
         picks=["mag", "stim"], scalings=dict(mag=2e-11, stim=1e3), show=True);

## 10. Summary

- **§2, §4 — Sidecar:** Cerca's `_meg.json` is not valid BIDS (`PowerLineFrequency: 0`, `RecordingType: "Measurement"`, empty descriptive fields, wrong channel counts). cMEG2FIF doesn't write a sidecar, but it puts the correct values into the FIF, and the future BIDS step will write the sidecar from there.
- **§5 — Channel types:** the FIF has four: `mag`, `ref_meg`, `stim`, `misc`.
- **§6 — Bad channels:** `info['bads']` starts empty unless channels were marked in Cerca; finding bad channels is your job.
- **§7 — Conversion log:** records provenance and every decision the converter made.
- **§8 — Peripherals:** the peripherals file maps BNC inputs to named button and analog channels; `STI101` holds VPixx codes and `STI_BTN` holds button identity as bits.